# 04. Bandingkan tracker di video CCTV (aplikasi Gradio)

Notebook ini membuka aplikasi **Bandingkan Tracker** di browser:

1. **Pilih video** dari daftar (video PT SRP di `~/PPE - PT SRP/Dataset_Momen_PPE_Anonim` dan `data/videos/`), atau upload video lain.
2. **Pilih tracker** yang mau dibandingkan, FPS proses, dan durasi.
3. Tekan **Proses**. Grid video tampil langsung: video asli, deteksi mentah, lalu satu panel per tracker. Di atasnya tampil **FPS yang sedang berjalan**, progres, dan waktu detektor.
4. Setelah selesai (atau ditekan **Hentikan**), tekan **Unduh video hasil**.

Detektor D-FINE jalan sekali per frame yang diproses, lalu deteksi yang sama dibagikan ke semua tracker, jadi beda antar panel murni dari tracker.

**Tanpa notebook**, aplikasi yang sama bisa dibuka dari Terminal:
```bash
cd ~/Documents/GitHub/object-tracking
uv run tracker-compare        # lalu buka http://127.0.0.1:7861
```

In [ ]:
# Cek Python yang dipakai notebook. Harus .venv milik repo ini.
import sys, importlib.util
print("Python:", sys.executable)
missing = [m for m in ('gradio', 'onnxruntime', 'cv2', 'pandas', 'ultralytics') if importlib.util.find_spec(m) is None]
if ".venv" not in sys.executable or missing:
    raise RuntimeError(
        "Notebook belum memakai .venv proyek" + (f" (paket tidak ada: {', '.join(missing)})" if missing else "") + ".\n"
        "Jalankan `uv sync` di folder repo, lalu di VS Code: klik nama kernel di kanan atas > Select Another Kernel > "
        "Python Environments > .venv (object-tracking), atau Enter interpreter path: "
        "~/Documents/GitHub/object-tracking/.venv/bin/python"
    )
print("ok, semua paket ada")

## Buka aplikasi

In [ ]:
import importlib
from pathlib import Path

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))
from object_tracking import live_compare, compare_app
importlib.reload(live_compare)     # kernel yang sudah lama jalan bisa masih memegang versi lama modul ini
importlib.reload(compare_app)

# ===================== ubah di sini =====================
VIDEO_DIRS = [Path.home() / "PPE - PT SRP" / "Dataset_Momen_PPE_Anonim",   # video PT SRP
              REPO / "data" / "videos"]                                     # video yang di-upload lewat aplikasi
DET_MODEL  = REPO / "models" / "dfine-n-1248x704-commercial-v3.onnx"
PORT       = 7861
# ========================================================

for p in VIDEO_DIRS + [DET_MODEL]:
    print(f"{'ok' if p.exists() else 'TIDAK ADA':9s} {p}")

if "demo" in globals():            # sel dijalankan ulang: tutup aplikasi lama dulu supaya port bebas
    demo.close()
demo = compare_app.build_app(VIDEO_DIRS, DET_MODEL)
compare_app.launch(demo, port=PORT, block=False, inbrowser=True, inline=False)
print(f"\nAplikasi jalan di http://127.0.0.1:{PORT} (tab browser terbuka otomatis).")
print("Video hasil tersimpan di", (REPO / "outputs" / "compare").relative_to(REPO))

## Cara membaca

**FPS di bagian atas:**
- **Sekarang ... frame/detik**: kecepatan proses nyata di mesin ini (detektor, semua tracker, dan menggambar grid). Kalau angkanya di atas FPS proses, mesin ini lebih cepat dari real-time untuk 1 kamera.
- **Batas 1 kamera di mesin ini**: 1000 / (waktu detektor + tracker paling lambat). Ini batas atas tanpa biaya decode dan menggambar.
- Supaya angka FPS menggambarkan kemampuan mesin, biarkan **Batasi ke kecepatan video asli** mati (di Pengaturan lanjutan). Nyalakan kalau ingin menonton dengan kecepatan normal.

**Panel:**
- Warna dan nomor = ID tracker. Orang yang sama seharusnya memegang warna yang sama sepanjang video. Garis = jejak gerak 3 detik terakhir.
- **Label kuning** = ID baru yang muncul di tengah video. Kalau orangnya sudah terlihat sebelumnya, itu ID switch.
- **Box hanya sudut** = posisi tebakan di frame tanpa deteksi (misalnya "Lite tuned det/2").

**Tabel ringkasan:** `ID dibuat` adalah jumlah ID berbeda yang benar-benar tampil, makin dekat ke jumlah orang yang sebenarnya lewat makin baik. `ID baru di tengah` adalah kandidat ID switch, makin kecil makin baik.

## Tutup aplikasi

In [ ]:
demo.close()
print("aplikasi ditutup")